<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_10_xgb_model/stage_10_xgb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_10 - XGBOOST - T2 SEQ2ONE**

# **SETUP COMÚN DEL PIPELINE DE ENTRENAMIENTO**

## **1. Imports**

In [1]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
import pandas as pd
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-04-16 01:55:34,698 | INFO | Environment initialized


In [2]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True  # <-- CLAVE en notebooks
)

## **2. Acceso a drive**

In [3]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

Mounted at /content/drive


2026-04-16 01:55:57,885 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


## **3. Carga de datasets origen**

### **3.1. Rutas de datasets escalados**

In [4]:
# ============================================================
# ENTRADAS: T2 SPLITS
# ============================================================
IN_T2_SUMMARY = DRIVE_DIR / Path(os.environ.get("IN_T2_SUMMARY", "data/04_features/mnq_t2_summary.json"))
IN_T2_SPLITS_SUMMARY = DRIVE_DIR / Path(os.environ.get("IN_T2_SPLITS_SUMMARY", "data/05_splits/splits_summary.json"))
IN_T2_PARQUET_TRAIN = DRIVE_DIR / Path(os.environ.get("IN_T2_PARQUET_TRAIN", "data/05_splits/mnq_t2_train.parquet"))
IN_T2_PARQUET_VALID = DRIVE_DIR / Path(os.environ.get("IN_T2_PARQUET_VALID", "data/05_splits/mnq_t2_valid.parquet"))
IN_T2_PARQUET_TEST = DRIVE_DIR / Path(os.environ.get("IN_T2_PARQUET_TEST", "data/05_splits/mnq_t2_test.parquet"))

# ============================================================
# ENTRADAS: DATASETS ESCALADOS
# ============================================================
IN_T2_TRAIN_Z = DRIVE_DIR / Path(os.environ.get("IN_T2_TRAIN_Z", "data/06_scaled/mnq_t2_train_z.parquet"))
IN_T2_VALID_Z = DRIVE_DIR / Path(os.environ.get("IN_T2_VALID_Z", "data/06_scaled/mnq_t2_valid_z.parquet"))
IN_T2_TEST_Z = DRIVE_DIR / Path(os.environ.get("IN_T2_TEST_Z", "data/06_scaled/mnq_t2_test_z.parquet"))
IN_T2_SCALER = DRIVE_DIR / Path(os.environ.get("IN_T2_SCALER", "data/06_scaled/scaler_t2.pkl"))
IN_T2_SCALER_META = DRIVE_DIR / Path(os.environ.get("IN_T2_SCALER_META", "data/06_scaled/scaler_meta_t2.json"))

### **3.2. Carga de datasets escalados**

In [5]:
# ============================================================
# CARGA DE DATASETS T2
# ============================================================

# -------------------------------
# 1. Splits originales
# -------------------------------
#df_train = pd.read_parquet(IN_T2_PARQUET_TRAIN)
#df_valid = pd.read_parquet(IN_T2_PARQUET_VALID)
#df_test  = pd.read_parquet(IN_T2_PARQUET_TEST)

#logging.info("Splits T2 cargados")
#logging.info(f"TRAIN: {df_train.shape}")
#logging.info(f"VALID: {df_valid.shape}")
#logging.info(f"TEST : {df_test.shape}")


# -------------------------------
# 2. Datasets escalados
# -------------------------------
df_train_z = pd.read_parquet(IN_T2_TRAIN_Z)
df_valid_z = pd.read_parquet(IN_T2_VALID_Z)
df_test_z  = pd.read_parquet(IN_T2_TEST_Z)

logging.info("Datasets escalados cargados")
logging.info(f"TRAIN_Z: {df_train_z.shape}")
logging.info(f"VALID_Z: {df_valid_z.shape}")
logging.info(f"TEST_Z : {df_test_z.shape}")


# -------------------------------
# 3. Scaler + metadata
# -------------------------------
import joblib

scaler_t2 = joblib.load(IN_T2_SCALER)

with open(IN_T2_SCALER_META, "r") as f:
    scaler_meta_t2 = json.load(f)

logging.info(f"Scaler cargado: {type(scaler_t2).__name__}")
logging.info(f"Scaler meta keys: {list(scaler_meta_t2.keys())}")

2026-04-16 01:56:06,456 | INFO | Datasets escalados cargados
2026-04-16 01:56:06,457 | INFO | TRAIN_Z: (462966, 18)
2026-04-16 01:56:06,458 | INFO | VALID_Z: (99134, 18)
2026-04-16 01:56:06,459 | INFO | TEST_Z : (99645, 18)
2026-04-16 01:56:08,278 | INFO | Scaler cargado: StandardScaler
2026-04-16 01:56:08,279 | INFO | Scaler meta keys: ['scaled', 'scaler_class', 'scale_cols', 'temporal_order_validated', 'sorted_by']


## **4. Carga de ventanas X/y**

### **4.1. Rutas de ventanas `seq2one` y `scaler` para L=30**

In [6]:
# ================================
# Configuración de paths
# ================================

WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / os.environ.get(
    "WINDOWS_SEQ2ONE_DIR",
    "data/07_windows/seq2one/"
)

SCALERS_DIR = DRIVE_DIR / os.environ.get(
    "SCALERS_DIR",
    "data/06_scaled/"
)

# -------------------------------
# Validación de paths
# -------------------------------
for p in [WINDOWS_SEQ2ONE_DIR, SCALERS_DIR]:
    if not p.exists():
        logger.warning(f"[PATH] No existe: {p}")
    else:
        logger.info(f"[PATH] OK: {p}")


# ================================
# Configuración del experimento
# ================================

TARGET = "t2_dir_thr_90"   # ← simplificado (ya no lista)
WINDOW_SIZE = 30           # ← fijo
SPLITS = ["train", "valid", "test"]

FEATURES_T2 = [
    "regime_id",
    "roc_30",
    "roc_60",
    "stoch_k_30",
    "atr_norm_10",
]

logger.info("[CONFIG] Experimento cargado")
logger.info(f"[CONFIG] Target: {TARGET}")
logger.info(f"[CONFIG] Window size: {WINDOW_SIZE}")

2026-04-16 01:56:08,288 | INFO | [PATH] OK: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one
2026-04-16 01:56:08,290 | INFO | [PATH] OK: /content/drive/MyDrive/neural_profit/data/06_scaled
2026-04-16 01:56:08,291 | INFO | [CONFIG] Experimento cargado
2026-04-16 01:56:08,293 | INFO | [CONFIG] Target: t2_dir_thr_90
2026-04-16 01:56:08,294 | INFO | [CONFIG] Window size: 30


In [7]:
# ================================
# Construcción de paths (simple)
# ================================

WINDOWS_L30_DIR = WINDOWS_SEQ2ONE_DIR / f"L{WINDOW_SIZE}"

TRAIN_WINDOW_PATH = WINDOWS_L30_DIR / f"windows_{TARGET}_train.npz"
VALID_WINDOW_PATH = WINDOWS_L30_DIR / f"windows_{TARGET}_valid.npz"
TEST_WINDOW_PATH  = WINDOWS_L30_DIR / f"windows_{TARGET}_test.npz"

SCALER_T2_PATH = SCALERS_DIR / "scaler_t2.pkl"

logger.info("[PATHS] Construidos")
logger.info(f"TRAIN: {TRAIN_WINDOW_PATH}")
logger.info(f"VALID: {VALID_WINDOW_PATH}")
logger.info(f"TEST : {TEST_WINDOW_PATH}")
logger.info(f"SCALER: {SCALER_T2_PATH}")


# ================================
# Validación (clara y directa)
# ================================
paths = {
    "train": TRAIN_WINDOW_PATH,
    "valid": VALID_WINDOW_PATH,
    "test": TEST_WINDOW_PATH,
    "scaler": SCALER_T2_PATH,
}

missing = [k for k, v in paths.items() if not v.exists()]

if not missing:
    logger.info("[CHECK] Todos los archivos existen")
else:
    logger.warning(f"[CHECK] Faltan: {missing}")
    for k in missing:
        logger.warning(f"{k}: {paths[k]}")

2026-04-16 01:56:08,302 | INFO | [PATHS] Construidos
2026-04-16 01:56:08,303 | INFO | TRAIN: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_dir_thr_90_train.npz
2026-04-16 01:56:08,304 | INFO | VALID: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_dir_thr_90_valid.npz
2026-04-16 01:56:08,305 | INFO | TEST : /content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_dir_thr_90_test.npz
2026-04-16 01:56:08,306 | INFO | SCALER: /content/drive/MyDrive/neural_profit/data/06_scaled/scaler_t2.pkl
2026-04-16 01:56:09,070 | INFO | [CHECK] Todos los archivos existen


### **4.2. Cargar ventanas (*.npz)**

In [8]:
from pathlib import Path
from typing import Tuple
import numpy as np


def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar (T2 seq2one).

    Espera:
    - 'X': (n_samples, seq_len, n_features)
    - 'y' o 'Y': (n_samples,) o (n_samples, 1) o (n_samples, seq_len, 1)

    Retorna
    -------
    X : np.ndarray  -> (n_samples, seq_len, n_features)
    y : np.ndarray  -> (n_samples,)
    """

    # --------------------------------------------------
    # 1. Validación
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")

        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # copiar a memoria
        X = X.copy()
        y = y.copy()

    # --------------------------------------------------
    # 3. Ajuste de shape de y (CRÍTICO)
    # --------------------------------------------------
    # Caso (n_samples, 1)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.reshape(-1)

    # Caso (n_samples, seq_len, 1) -> tomar último valor
    elif y.ndim == 3:
        y = y[:, -1, 0]

    # Caso (n_samples, seq_len) -> tomar último valor
    elif y.ndim == 2 and y.shape[1] > 1:
        y = y[:, -1]

    # Validación final
    if y.ndim != 1:
        raise ValueError(f"y no es 1D después de procesamiento: shape={y.shape}")

    # --------------------------------------------------
    # 4. Logs útiles
    # --------------------------------------------------
    logger.info(f"Loaded: {path.name}")
    logger.info(f"X shape: {X.shape} | y shape: {y.shape}")

    return X, y

In [9]:
# ================================
# Carga directa de ventanas
# ================================

logger.info("[LOAD] Ventanas L=30 | T2 thr=90")

X_train, y_train = load_npz_windows(TRAIN_WINDOW_PATH)
X_valid, y_valid = load_npz_windows(VALID_WINDOW_PATH)
X_test,  y_test  = load_npz_windows(TEST_WINDOW_PATH)

logger.info("[DONE] Ventanas cargadas")

2026-04-16 01:56:09,085 | INFO | [LOAD] Ventanas L=30 | T2 thr=90
2026-04-16 01:56:10,844 | INFO | Loaded: windows_t2_dir_thr_90_train.npz
2026-04-16 01:56:10,845 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-16 01:56:11,602 | INFO | Loaded: windows_t2_dir_thr_90_valid.npz
2026-04-16 01:56:11,604 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-16 01:56:12,464 | INFO | Loaded: windows_t2_dir_thr_90_test.npz
2026-04-16 01:56:12,465 | INFO | X shape: (93990, 30, 5) | y shape: (93990,)
2026-04-16 01:56:12,466 | INFO | [DONE] Ventanas cargadas


In [10]:
# ================================
# Flatten para XGBoost
# ================================

X_train = X_train.reshape(X_train.shape[0], -1)
X_valid = X_valid.reshape(X_valid.shape[0], -1)
X_test  = X_test.reshape(X_test.shape[0], -1)

logger.info("[SHAPE] Datos listos para XGBoost")
logger.info(f"TRAIN: {X_train.shape}")
logger.info(f"VALID: {X_valid.shape}")
logger.info(f"TEST : {X_test.shape}")

2026-04-16 01:56:12,473 | INFO | [SHAPE] Datos listos para XGBoost
2026-04-16 01:56:12,475 | INFO | TRAIN: (436692, 150)
2026-04-16 01:56:12,476 | INFO | VALID: (93508, 150)
2026-04-16 01:56:12,477 | INFO | TEST : (93990, 150)


### **4.3. Cargar escalador (*.pkl)**

In [11]:
# --------------------------------------------------
# Función común: carga scaler compartido (T2)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib.

    Parámetros
    ----------
    path : Path
        Ruta al archivo scaler (.pkl)

    Retorna
    -------
    scaler : Any
        Objeto scaler (ej. StandardScaler, MinMaxScaler)
    """

    # --------------------------------------------------
    # 1. Validación de existencia
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    scaler = joblib.load(path)

    # --------------------------------------------------
    # 3. Validación básica (opcional pero útil)
    # --------------------------------------------------
    if not hasattr(scaler, "transform"):
        raise TypeError(f"El objeto cargado no es un scaler válido: {type(scaler)}")

    # --------------------------------------------------
    # 4. Logging
    # --------------------------------------------------
    logger.info(f"Scaler cargado: {path.name}")

    return scaler

In [12]:
scaler_t2 = load_scaler(SCALER_T2_PATH)

2026-04-16 01:56:12,495 | INFO | Scaler cargado: scaler_t2.pkl


## **5. Sanity Check**

Este bloque sirve para:

1. Validar que los datos están bien formados
    - Shapes correctos (X, y)
    - Sin NaN / inf
    - Consistencia entre splits
    
2. Detectar errores silenciosos
    - Targets mal alineados
    - Ventanas mal construidas
    - Clases faltantes (crítico en T2)

3. Ver distribución de clases
    - Muy importante en tu caso (clase 0 dominante)

In [13]:
from __future__ import annotations

from typing import Any, Optional, Tuple
import numpy as np
# ============================================================
# 2) Sanity check principal (seq2one clasificación T2)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one de clasificación.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como d_flat esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado).
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len), permite tomar y[:, -1].
        Por defecto False.
    """

    X = _as_numpy_array(X, name=f"X[{split_name}]")
    y = _as_numpy_array(y, name=f"y[{split_name}]")

    # --------------------------------------------------
    # Normalización de y
    # --------------------------------------------------
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        y = y[:, -1]

    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # --------------------------------------------------
    # Inferir modo y dimensiones de X
    # --------------------------------------------------
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # --------------------------------------------------
    # Validación básica de n_samples
    # --------------------------------------------------
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: "
            f"X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # --------------------------------------------------
    # Validación de shapes según modo
    # --------------------------------------------------
    if mode == "3d":
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, "
                f"recibido={seq_len}. X.shape={X.shape}"
            )

        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, "
                f"recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim o pase X en 3D."
            )

    # --------------------------------------------------
    # Diagnóstico de clases
    # --------------------------------------------------
    classes, counts = np.unique(y, return_counts=True)
    class_distribution = {
        str(cls): int(cnt) for cls, cnt in zip(classes, counts)
    }

    if len(classes) < 2:
        raise ValueError(
            f"{split_name}: y contiene menos de 2 clases únicas. "
            f"classes={classes.tolist()}"
        )

    # --------------------------------------------------
    # Salida informativa
    # --------------------------------------------------
    info = {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "n_classes": int(len(classes)),
        "classes": classes.tolist(),
        "class_distribution": class_distribution,
    }

    if verbose:
        print(
            f"[sanity_check_seq2one] {split_name} | "
            f"X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | n_classes={info['n_classes']} | "
            f"classes={info['classes']}"
        )

    return info

In [14]:
def run_sanity_checks_seq2one_simple(
    *,
    X_train, y_train,
    X_valid, y_valid,
    X_test,  y_test,
    tag: str = "t2_L30",
    expected_seq_len: int | None = None,
    expected_n_features: int | None = None,
    expected_flat_dim: int | None = None,
    verbose: bool = True,
):
    """
    Sanity check simplificado para seq2one.

    Soporta:
    - X 3D: (n, seq_len, n_features)
    - X 2D: (n, d_flat)

    Parámetros
    ----------
    expected_seq_len:
        Solo aplica a X 3D
    expected_n_features:
        Solo aplica a X 3D
    expected_flat_dim:
        Solo aplica a X 2D (ej: 150)
    """

    # --------------------------------------------------
    # Detectar automáticamente el modo (2D o 3D)
    # --------------------------------------------------
    X_sample = np.asarray(X_train)

    if X_sample.ndim == 3:
        mode = "3d"
    elif X_sample.ndim == 2:
        mode = "2d"
    else:
        raise ValueError(f"X_train tiene ndim inválido: {X_sample.shape}")

    # --------------------------------------------------
    # Ejecutar checks según modo
    # --------------------------------------------------
    if mode == "3d":
        kwargs = dict(
            expected_seq_len=expected_seq_len,
            expected_n_features=expected_n_features,
            expected_flat_dim=None,
        )
    else:
        kwargs = dict(
            expected_seq_len=None,
            expected_n_features=None,
            expected_flat_dim=expected_flat_dim,
        )

    out_train = sanity_check_seq2one(
        X_train, y_train,
        "train",
        **kwargs,
        verbose=verbose,
    )

    out_valid = sanity_check_seq2one(
        X_valid, y_valid,
        "valid",
        **kwargs,
        verbose=verbose,
    )

    out_test = sanity_check_seq2one(
        X_test, y_test,
        "test",
        **kwargs,
        verbose=verbose,
    )

    if verbose:
        print(f"\nOK SANITY CHECK | {tag} | mode={mode}")

    return {
        "mode": mode,
        "train": out_train,
        "valid": out_valid,
        "test": out_test,
    }

In [15]:
sanity = run_sanity_checks_seq2one_simple(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
    expected_flat_dim=150,  # 30 * 5
)

NameError: name '_as_numpy_array' is not defined

## **6. Métricas de clasificación T2**

In [16]:
# ================================
# Setup para importar módulos del proyecto
# ================================

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

# ================================
# Imports de métricas (T2 clasificación)
# ================================

from metrics.classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

# ================================
# Utilidad: métricas → DataFrame
# ================================

import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte métricas de clasificación T2 en una fila de DataFrame.
    """

    m = metrics["metrics"]

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "n_samples": m["n_samples"],

        # principales
        "balanced_accuracy": m["balanced_accuracy"],
        "f1_macro": m["f1_macro"],
        "f1_weighted": m["f1_weighted"],

        # complementarias
        "accuracy": m["accuracy"],
        "precision_macro": m["precision_macro"],
        "recall_macro": m["recall_macro"],

        # baseline
        "balanced_accuracy_naive": m.get("balanced_accuracy_naive"),
        "bal_acc_gain_vs_naive": m.get("balanced_accuracy_gain_vs_naive"),
    }])

logger.info("Métricas T2 + utilidades cargadas")

2026-04-16 01:56:22,983 | INFO | Métricas T2 + utilidades cargadas


## **7. Persistencia de métricas (tracking de experimentos)**

In [17]:
# ================================
# Carga de métricas (si existen)
# ================================
def load_classification_metrics_if_exists(
    *,
    name: str,
    base_dir: Path = DRIVE_DIR / "xgb_final",
) -> pd.DataFrame:

    path = base_dir / f"classification_{name}_metrics.parquet"

    if path.exists():
        logger.info(f"[LOAD] Métricas encontradas: {path}")
        df = pd.read_parquet(path)
        logger.info(f"[LOAD] Shape: {df.shape}")
        return df

    logger.info(f"[LOAD] No existen métricas previas para: {name}")
    return pd.DataFrame()

In [18]:
# ================================
# Guardado de métricas
# ================================
def save_classification_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: Path = DRIVE_DIR / "xgb_final",
) -> Path:

    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_{name}_metrics.parquet"

    df_metrics.to_parquet(out_path, index=False)

    logger.info(f"[SAVE] Métricas guardadas en: {out_path}")
    logger.info(f"[SAVE] Shape: {df_metrics.shape}")

    return out_path

Como usarlo:

```python
name = "xgboost_t2_L30_thr90"

df_prev = load_classification_metrics_if_exists(name=name)

if df_prev.empty:
    logger.info("No hay métricas previas → continuar entrenamiento")
else:
    logger.info("Ya existen métricas → podrías saltar entrenamiento")
```



## **8. Gestión de dispositivo y memoria**

In [19]:
# ================================
# Device y limpieza de memoria
# ================================

def get_torch_device():
    """
    Retorna el device para modelos PyTorch.
    """
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    return device


def clear_torch_memory() -> None:
    """
    Limpia memoria Python y, si existe CUDA, libera caché GPU.
    Útil entre entrenamientos de modelos PyTorch.
    """
    import gc

    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

## **9. Reproducibilidad**

In [20]:
# ================================
# Reproducibilidad
# ================================

def set_seeds(seed: int = 42) -> None:
    """
    Fija semillas para reproducibilidad en:
    - Python
    - NumPy
    - PyTorch (si está disponible)
    """

    import random
    import numpy as np
    import os

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # ---- PyTorch (si está instalado)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # determinismo (más lento pero reproducible)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    except ImportError:
        pass  # no hay PyTorch, ignorar

    logger.info(f"Seeds fijadas en {seed}")


# Aplicación
SEED = 42
set_seeds(SEED)

2026-04-16 01:56:31,500 | INFO | Seeds fijadas en 42


# **10. Modelo XGBoost**

In [21]:
param_grid_xgb_final = {
    # ---------------------------
    # Núcleo (confirmado)
    # ---------------------------
    "n_estimators": [200],
    "max_depth": [3],
    "learning_rate": [0.03],

    # ---------------------------
    # Subsampling (robustez)
    # ---------------------------
    "subsample": [0.8],
    "colsample_bytree": [0.8],

    # ---------------------------
    # Regularización
    # ---------------------------
    "reg_lambda": [10.0],
    "reg_alpha": [0.0],

    # ---------------------------
    # Micro-ajuste (único rango abierto)
    # ---------------------------
    "min_child_weight": [1], #, 2, 3],

    # ---------------------------
    # Sin impacto
    # ---------------------------
    "gamma": [0.0],
}

## **10.1. Función `train_xgboost_final`**

In [22]:
from xgboost import XGBClassifier
import numpy as np


def train_xgboost_final(
    *,
    X_train, y_train,
    X_valid, y_valid,
    X_test,  y_test,
    random_state: int = 42,
    n_jobs: int = -1,
    class_weight: str | dict | None = "balanced",
    use_gpu: bool = True,
    verbose: bool = False,
):
    """
    Modelo XGBoost final para:
    - window_size = 30
    - target = t2_dir_thr_90

    Inputs deben estar en formato 2D (flatten).
    """

    # =========================
    # 1. VALIDACIONES
    # =========================
    if X_train.ndim != 2:
        raise ValueError(f"X_train debe ser 2D (flatten). Recibido: {X_train.shape}")

    # =========================
    # 2. ENCODE LABELS
    # =========================
    classes_ = np.sort(np.unique(y_train))

    if set(np.unique(y_valid)) - set(classes_):
        raise ValueError("VALID contiene clases no vistas en TRAIN")

    if set(np.unique(y_test)) - set(classes_):
        raise ValueError("TEST contiene clases no vistas en TRAIN")

    class_to_idx = {cls: idx for idx, cls in enumerate(classes_)}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

    y_train_enc = np.array([class_to_idx[y] for y in y_train], dtype=np.int32)
    num_class = len(classes_)

    # =========================
    # 3. SAMPLE WEIGHT
    # =========================
    sample_weight = None
    weights_by_idx = None

    if class_weight == "balanced":
        counts = np.bincount(y_train_enc, minlength=num_class)
        total = counts.sum()

        weights_by_idx = {
            idx: total / (num_class * count)
            for idx, count in enumerate(counts)
        }

        sample_weight = np.array(
            [weights_by_idx[idx] for idx in y_train_enc],
            dtype=np.float32,
        )

    elif isinstance(class_weight, dict):
        weights_by_idx = {
            class_to_idx[cls]: weight
            for cls, weight in class_weight.items()
            if cls in class_to_idx
        }

        sample_weight = np.array(
            [weights_by_idx.get(idx, 1.0) for idx in y_train_enc],
            dtype=np.float32,
        )

    # =========================
    # 4. MODELO (HIPERPARÁMETROS FINALES)
    # =========================
    device = "cuda" if use_gpu else "cpu"

    model = XGBClassifier(
        objective="multi:softprob",
        num_class=num_class,

        # 🔥 PARAMS FINALES
        n_estimators=200,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=1,
        gamma=0.0,
        reg_alpha=0.0,
        reg_lambda=10.0,

        tree_method="hist",
        device=device,
        random_state=random_state,
        n_jobs=n_jobs,
        eval_metric="mlogloss",
        verbosity=1 if verbose else 0,
    )

    # =========================
    # 5. TRAIN
    # =========================
    model.fit(
        X_train,
        y_train_enc,
        sample_weight=sample_weight,
    )

    # =========================
    # 6. PREDICT VALID
    # =========================
    y_pred_valid_enc = model.predict(X_valid)
    y_pred_valid = np.array([idx_to_class[int(y)] for y in y_pred_valid_enc])
    y_proba_valid = model.predict_proba(X_valid)

    # =========================
    # 7. PREDICT TEST
    # =========================
    y_pred_test_enc = model.predict(X_test)
    y_pred_test = np.array([idx_to_class[int(y)] for y in y_pred_test_enc])
    y_proba_test = model.predict_proba(X_test)

    return {
        "model": model,
        "classes_": classes_,
        "class_to_idx": class_to_idx,
        "idx_to_class": idx_to_class,
        "weights_by_idx": weights_by_idx,
        "y_true_valid": y_valid,
        "y_pred_valid": y_pred_valid,
        "y_proba_valid": y_proba_valid,
        "y_true_test": y_test,
        "y_pred_test": y_pred_test,
        "y_proba_test": y_proba_test,
    }

Como usarla:

```python
result = train_xgboost_final(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
)
```



## **10.2. Función `evaluate_xgboost_final`**

In [23]:
def evaluate_xgboost_final(
    *,
    result: dict,
    model_name: str = "xgboost_final",
):
    """
    Evalúa el modelo XGBoost final en VALID y TEST.

    Retorna un DataFrame con ambas evaluaciones.
    """

    rows = []

    for split in ["valid", "test"]:

        # -------------------------------
        # Selección de datos
        # -------------------------------
        y_true = result[f"y_true_{split}"]
        y_pred = result[f"y_pred_{split}"]

        # -------------------------------
        # Métricas
        # -------------------------------
        metrics = compute_classification_metrics(
            y_true=y_true,
            y_pred=y_pred,
            model_name=model_name,
            split=split,
            target="t2_dir_thr_90",
            labels=[-1, 0, 1],
        )

        # -------------------------------
        # DataFrame
        # -------------------------------
        df_row = metrics_to_df(
            metrics,
            model=model_name,
            split=split,
            window_size=30,
            target="t2_dir_thr_90",
        )

        # -------------------------------
        # Metadata
        # -------------------------------
        df_row["horizon_min"] = 90
        df_row["class_weight_mode"] = "balanced"

        # Hiperparámetros finales
        df_row["n_estimators"] = 200
        df_row["max_depth"] = 3
        df_row["learning_rate"] = 0.03
        df_row["subsample"] = 0.8
        df_row["colsample_bytree"] = 0.8
        df_row["min_child_weight"] = 1
        df_row["gamma"] = 0.0
        df_row["reg_alpha"] = 0.0
        df_row["reg_lambda"] = 10.0

        rows.append(df_row)

    return pd.concat(rows, ignore_index=True)

Como usarla:

```python
result = train_xgboost_final(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
)

df_metrics = evaluate_xgboost_final(result=result)
```

## **10.3. Función `train_or_load_xgboost_final`**

In [24]:
import joblib
import json


def train_or_load_xgboost_final(
    *,
    X_train, y_train,
    X_valid, y_valid,
    X_test,  y_test,
    model_name: str = "xgboost_t2_L30_thr90",
    models_dir: Path = DRIVE_DIR / "xgb_final",
    use_gpu: bool = True,
):
    """
    Entrena o carga el modelo XGBoost final, evalúa VALID/TEST
    y guarda modelo, metadata y métricas.
    """

    models_dir.mkdir(parents=True, exist_ok=True)

    model_path = models_dir / f"{model_name}.pkl"
    meta_path = models_dir / f"{model_name}_meta.json"
    metrics_path = models_dir / f"classification_{model_name}_metrics.parquet"

    meta = {
        "model_name": model_name,
        "window_size": 30,
        "target": "t2_dir_thr_90",
        "horizon_min": 90,
        "features": FEATURES_T2,
        "n_estimators": 200,
        "max_depth": 3,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 1,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 10.0,
        "class_weight": "balanced",
    }

    # =====================================================
    # 1. SI YA EXISTE → CARGAR
    # =====================================================
    if model_path.exists():
        logger.info(f"[LOAD] Modelo existente encontrado: {model_path}")
        model = joblib.load(model_path)

        if meta_path.exists():
            with open(meta_path, "r") as f:
                meta = json.load(f)

        df_metrics = pd.read_parquet(metrics_path) if metrics_path.exists() else pd.DataFrame()

        return {
            "model": model,
            "meta": meta,
            "df_metrics": df_metrics,
            "loaded": True,
        }

    # =====================================================
    # 2. ENTRENAR
    # =====================================================
    logger.info("[TRAIN] Entrenando modelo XGBoost final...")

    result = train_xgboost_final(
        X_train=X_train,
        y_train=y_train,
        X_valid=X_valid,
        y_valid=y_valid,
        X_test=X_test,
        y_test=y_test,
        use_gpu=use_gpu,
    )

    model = result["model"]

    # =====================================================
    # 3. EVALUAR
    # =====================================================
    df_metrics = evaluate_xgboost_final(
        result=result,
        model_name=model_name,
    )

    # =====================================================
    # 4. GUARDAR MODELO
    # =====================================================
    joblib.dump(model, model_path)
    logger.info(f"[SAVE] Modelo guardado en: {model_path}")

    # =====================================================
    # 5. GUARDAR METADATA
    # =====================================================
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=4)

    logger.info(f"[SAVE] Metadata guardada en: {meta_path}")

    # =====================================================
    # 6. GUARDAR MÉTRICAS
    # =====================================================
    df_metrics.to_parquet(metrics_path, index=False)
    logger.info(f"[SAVE] Métricas guardadas en: {metrics_path}")

    return {
        "model": model,
        "meta": meta,
        "result": result,
        "df_metrics": df_metrics,
        "loaded": False,
    }

## **10.4. Entrenamiento de modelo XGBoost final**

In [25]:
artifact = train_or_load_xgboost_final(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
)

2026-04-16 01:56:31,979 | INFO | [LOAD] Modelo existente encontrado: /content/drive/MyDrive/neural_profit/xgb_final/xgboost_t2_L30_thr90.pkl


In [26]:
xgb_metrics=artifact['df_metrics']

# **11. Evaluación de comportamiento real del modelo XGB**

Vamos a establecer un **plan de análisis del comportamiento real del modelo**, pasando de métricas globales a señales operables. Eso encaja con el flujo del libro: primero evaluar el modelo fuera de muestra y luego traducir sus señales en decisiones y reglas de trading.  

1. **Reconstruir el set de predicciones de test**

   * Armar una tabla base con:

     * `y_true`
     * `y_pred`
     * probabilidades por clase
     * `confidence`
   * Además unirla con metadatos del split test:

     * `date`
     * `minute_of_day`
     * timestamp o índice temporal equivalente

2. **Validar desempeño real a nivel de predicción**

   * Confirmar de nuevo, pero ya sobre la tabla final:

     * balanced accuracy
     * f1 macro
     * accuracy
   * Agregar:

     * matriz de confusión
     * accuracy por clase
   * Objetivo: ver **dónde acierta y dónde falla**, no solo el promedio global. Esto es importante porque en trading la calidad del diagnóstico del modelo y de sus errores es clave para iterar de forma útil.

3. **Definir formalmente qué es una señal de entrada**

   * Señal long: `y_pred = 1`
   * Señal short: `y_pred = -1`
   * No señal: `y_pred = 0`
   * Luego decidir si las entradas se filtran por confianza mínima o no

4. **Construir una medida de confianza**

   * Opción recomendada:

     * `confidence = max(probabilidades de clase)`
   * También guardar:

     * `proba_down`
     * `proba_neutral`
     * `proba_up`
   * Así podrás distinguir entre:

     * señal débil
     * señal intermedia
     * señal fuerte

5. **Analizar cantidad total de señales**

   * Cuántas predicciones totales hay en test
   * Cuántas son:

     * long
     * short
     * neutral
   * Qué porcentaje del total representan
   * Esto responde si el modelo realmente genera pocas oportunidades o demasiadas

6. **Analizar señales por nivel de confianza**

   * Separar por umbrales, por ejemplo:

     * sin filtro
     * `confidence >= 0.50`
     * `confidence >= 0.60`
     * `confidence >= 0.70`
   * Para cada umbral:

     * cantidad de señales
     * precisión de las señales
   * Aquí aparecerá el trade-off central:

     * menos señales, pero más confiables

7. **Analizar señales por día**

   * Contar cuántas entradas se generan por jornada
   * Separar:

     * total señales por día
     * long por día
     * short por día
   * Luego resumir:

     * promedio
     * mediana
     * mínimo
     * máximo
   * Esto te dirá si el modelo opera de forma pareja o concentrada

8. **Analizar en qué momento del día se generan más señales**

   * Agrupar por `minute_of_day`
   * Medir:

     * número de señales
     * confianza promedio
     * precisión por franja horaria
   * Esto es muy valioso en intradía, porque puede revelar que el modelo funciona mejor en apertura, tramo medio o cierre

9. **Analizar calidad de señal por horario**

   * No solo cuántas señales aparecen en cada momento
   * También:

     * qué tan confiables son
     * qué tan precisas son
   * Porque puede ocurrir que una franja genere muchas señales, pero de baja calidad

10. **Construir el resumen operativo final**

* Responder con evidencia:

  * cuántas señales genera el modelo
  * en qué horarios aparecen más
  * cuáles son más confiables
  * qué umbral de confianza parece razonable
* Ese resumen será la base para pasar luego a reglas de trading y backtest, que es justamente el siguiente paso natural del workflow ML4T.

Mi recomendación concreta es desarrollar esto en este orden:

**Paso 1:** tabla completa de predicciones de test
**Paso 2:** definición de confianza
**Paso 3:** conteo de señales totales
**Paso 4:** señales por día
**Paso 5:** señales por momento del día
**Paso 6:** precisión según confianza y horario
**Paso 7:** conclusiones operativas

Me parece el orden más limpio porque primero construyes la base correcta y luego haces análisis cada vez más útiles para trading.


## **11.1. Reconstruir el set de predicciones de test**


### **1. Estructura objetivo de la tabla**



Necesitas un `DataFrame` con esta estructura mínima:

* `y_true` → valor real
* `y_pred` → clase predicha
* `proba_-1`
* `proba_0`
* `proba_1`
* `confidence` → max(probabilidades)
* `date`
* `minute_of_day`

Opcional pero útil:

* `timestamp` (si lo tienes)

### **2. Construcción paso a paso**



Asumiendo que ya tienes:

* `X_test`
* `y_test`
* `model` entrenado
* y acceso al índice temporal original

In [27]:
import numpy as np
import pandas as pd
import logging

# =========================================
# Configuración de logging
# =========================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger(__name__)

original_test_df=df_test_z.copy()
model = artifact['model']

#### Paso 1: Predicciones del modelo (inferencia sobre test)



En este paso se ejecuta el modelo ya entrenado sobre el conjunto de test, es decir, sobre datos que el modelo nunca vio durante el entrenamiento. El objetivo es obtener las decisiones del modelo y su nivel de confianza en cada una.

Se generan dos salidas principales:

- y_pred = model.predict(X_test)  
  Representa la decisión final del modelo para cada observación.  
  En el caso del target T2:
  - -1 → el modelo predice caída
  - 0 → el modelo predice zona neutral (no operar)
  - 1 → el modelo predice subida  

  Esta es la señal directa que usarías para tomar decisiones de trading.

- y_proba = model.predict_proba(X_test)  
  Representa la probabilidad asignada a cada clase.  
  Para cada observación, devuelve un vector con la probabilidad de cada escenario.

  Ejemplo:
  [0.10, 0.75, 0.15]

  Esto significa:
  - 10% probabilidad de bajar (-1)
  - 75% probabilidad de neutral (0)
  - 15% probabilidad de subir (1)

Este paso es clave porque transforma el modelo en algo operativo: pasa de ser un modelo entrenado a generar decisiones concretas en el tiempo.

En términos de trading:
- y_pred indica qué harías (long, short o no operar)
- y_proba indica qué tan seguro está el modelo de esa decisión

Un punto importante es que no todas las predicciones tienen la misma calidad. Por ejemplo:

Caso 1: [0.34, 0.33, 0.33] → alta incertidumbre  
Caso 2: [0.10, 0.80, 0.10] → alta confianza  

Aunque ambas podrían dar la misma clase final, la segunda es mucho más confiable.  
Por eso más adelante se utilizará una medida de “confidence” para filtrar señales.

En resumen, este paso responde a la pregunta:
“¿Qué predice el modelo y con qué nivel de confianza, punto a punto en el conjunto de test?”

In [28]:
# =========================================
# 1. Predicciones del modelo
# =========================================

logger.info("Paso 1: Generando predicciones del modelo...")

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

logger.info(f"Predicciones generadas | shape y_pred: {y_pred.shape} | shape y_proba: {y_proba.shape}")

2026-04-16 01:56:36,543 | INFO | Paso 1: Generando predicciones del modelo...
2026-04-16 01:56:36,923 | INFO | Predicciones generadas | shape y_pred: (93990,) | shape y_proba: (93990, 3)


#### Paso 2: Construcción del DataFrame base



En este paso se construye una tabla (DataFrame) que contiene, para cada observación del conjunto de test, el valor real y la predicción del modelo.

Se crean dos columnas principales:

- y_true  
  Representa el valor real observado (la verdad del mercado).  
  Es decir, lo que efectivamente ocurrió:
  - -1 → el precio bajó más allá del umbral
  - 0 → no hubo movimiento significativo
  - 1 → el precio subió más allá del umbral  

- y_pred  
  Representa la predicción del modelo para esa misma observación.  
  Es la decisión que el modelo habría tomado en ese momento.

El resultado es una tabla alineada fila a fila:

Cada fila representa un instante en el tiempo, donde:
- y_true → qué pasó realmente
- y_pred → qué dijo el modelo que iba a pasar

Este paso es fundamental porque permite comparar directamente predicción vs realidad, lo cual es la base de:

- calcular métricas
- analizar errores
- evaluar desempeño real del modelo
- construir señales de trading

En este punto todavía no se tiene información de confianza ni probabilidades, solo la comparación directa entre realidad y predicción.

En resumen, este paso responde a la pregunta:
“Para cada instante del test, ¿qué pasó realmente y qué predijo el modelo?”

In [29]:
# =========================================
# 2. Construcción base
# =========================================

logger.info("Paso 2: Construyendo DataFrame base...")

df_pred = pd.DataFrame({
    "y_true": y_test,
    "y_pred": y_pred,
})

# -----------------------------
# Corregir etiquetas predichas
# de espacio codificado -> clases reales
# -----------------------------
idx_to_class = {0: -1, 1: 0, 2: 1}
df_pred["y_pred"] = df_pred["y_pred"].map(idx_to_class)

logger.info(f"DataFrame base creado | shape: {df_pred.shape}")

2026-04-16 01:56:36,930 | INFO | Paso 2: Construyendo DataFrame base...
2026-04-16 01:56:36,961 | INFO | DataFrame base creado | shape: (93990, 2)


#### Paso 3: Probabilidades por clase


En este paso se agregan al DataFrame las probabilidades que el modelo asigna a cada clase para cada observación.

El modelo no solo entrega una predicción final (`y_pred`), sino también un vector de probabilidades (`y_proba`) que indica qué tan probable considera cada posible resultado.

Primero se obtienen las clases que el modelo reconoce:

- model.classes_

Esto asegura que el orden de las probabilidades coincida correctamente con cada clase.

Luego, para cada clase, se crea una nueva columna:

- proba_-1 → probabilidad de caída
- proba_0 → probabilidad de zona neutral
- proba_1 → probabilidad de subida

Cada fila queda entonces enriquecida con información más completa:

- y_true → lo que realmente ocurrió
- y_pred → lo que el modelo decidió
- proba_* → qué tan probable consideraba cada escenario

Este paso es clave porque permite pasar de una predicción “discreta” (solo una clase) a una visión probabilística del modelo.

Esto es especialmente importante en trading, porque dos predicciones iguales pueden tener niveles de confianza muy distintos. Por ejemplo:

- Caso A: proba_1 = 0.40 → señal débil  
- Caso B: proba_1 = 0.80 → señal fuerte  

Ambos casos podrían dar y_pred = 1, pero no deberían tratarse igual.

Estas probabilidades son la base para:

- calcular la “confidence” del modelo
- filtrar señales de baja calidad
- priorizar decisiones más confiables

En resumen, este paso responde a la pregunta:
“¿Qué tan probable considera el modelo cada posible resultado en cada instante?”

In [30]:
# =========================================
# 3. Probabilidades por clase
# =========================================

logger.info("Paso 3: Agregando probabilidades por clase...")

classes = model.classes_
logger.info(f"Clases detectadas: {classes}")

class_mapping = {0: -1, 1: 0, 2: 1}

for i, cls in enumerate(classes):
    real_cls = class_mapping[cls]
    df_pred[f"proba_{real_cls}"] = y_proba[:, i]

logger.info("Probabilidades agregadas correctamente")


2026-04-16 01:56:36,968 | INFO | Paso 3: Agregando probabilidades por clase...
2026-04-16 01:56:36,969 | INFO | Clases detectadas: [0 1 2]
2026-04-16 01:56:36,972 | INFO | Probabilidades agregadas correctamente


#### Paso 4: Cálculo de la confidence


En este paso se calcula una medida de “confianza” del modelo para cada predicción.

La idea es simple: de todas las probabilidades que el modelo asigna a las clases, se toma la mayor.

confidence = max(probabilidades)

Por ejemplo:

- [0.10, 0.75, 0.15] → confidence = 0.75  
- [0.34, 0.33, 0.33] → confidence = 0.34  

Esto representa qué tan convencido está el modelo de su decisión final (`y_pred`).

Interpretación:

- confidence alta (cercana a 1) → el modelo está seguro  
- confidence media (~0.5–0.6) → señal moderada  
- confidence baja (~0.33) → modelo indeciso (casi aleatorio)  

Este paso es clave porque permite diferenciar entre:

- predicciones fuertes (potencialmente operables)
- predicciones débiles (probablemente ruido)

En trading, esto es fundamental, porque no todas las predicciones deberían convertirse en operaciones. La confidence permite:

- filtrar señales de baja calidad
- reducir operaciones innecesarias
- concentrarse en escenarios con mayor probabilidad

Además, se calculan estadísticas básicas:

- min → peor caso de confianza
- mean → nivel promedio de seguridad del modelo
- max → mejor caso (predicciones muy claras)

En resumen, este paso responde a la pregunta:
“¿Qué tan seguro está el modelo de cada una de sus predicciones?”

In [31]:
# =========================================
# 4. Confidence
# =========================================

logger.info("Paso 4: Calculando confidence...")

df_pred["confidence"] = y_proba.max(axis=1)

logger.info(
    f"Confidence calculado | min: {df_pred['confidence'].min():.4f} | "
    f"mean: {df_pred['confidence'].mean():.4f} | "
    f"max: {df_pred['confidence'].max():.4f}"
)



2026-04-16 01:56:36,977 | INFO | Paso 4: Calculando confidence...
2026-04-16 01:56:36,985 | INFO | Confidence calculado | min: 0.3336 | mean: 0.4914 | max: 0.9039


#### Paso 5: Alineación temporal correcta de las predicciones


En este paso ya no se puede usar directamente todo el dataframe original de test (`df_test_z`), porque el modelo no genera una predicción por cada fila original, sino una predicción por cada ventana válida construida.

Como el tamaño de ventana es `L=30`, cada jornada pierde sus primeras `29` filas al momento de crear muestras seq2one. Esto ocurre porque una predicción solo puede generarse cuando ya existen 30 observaciones consecutivas disponibles.

Por lo tanto, las marcas temporales que deben asociarse a las predicciones no son todas las del dataset original, sino únicamente aquellas filas que corresponden al final de cada ventana.

Si las ventanas fueron construidas por jornada, sin cruzar días, entonces la alineación correcta debe hacerse también por jornada. Para cada día, se deben conservar solo las filas desde la posición `L-1` en adelante.

De esta manera, cada predicción queda correctamente asociada al instante temporal real al que pertenece.

En resumen, este paso corrige la correspondencia entre:

- las predicciones generadas por el modelo
- y la fila temporal exacta del dataset original que representa el final de cada ventana

In [32]:
# =========================================
# 5. Agregar información temporal
# =========================================

L = 30

logger.info("Paso 5: Reconstruyendo alineación temporal desde df_test_z...")

df_test_time = (
    df_test_z
    .groupby("date", group_keys=False)[df_test_z.columns]
    .apply(lambda x: x.iloc[L-1:])
    .reset_index(drop=True)
)

logger.info(
    f"Filas df_test_z original: {len(df_test_z)} | "
    f"filas alineadas para ventanas: {len(df_test_time)} | "
    f"filas df_pred: {len(df_pred)}"
)

if len(df_test_time) != len(df_pred):
    raise ValueError(
        f"No coincide la alineación temporal: "
        f"df_test_time={len(df_test_time)} vs df_pred={len(df_pred)}"
    )

df_pred["date"] = df_test_time["date"].values
df_pred["minute_of_day"] = df_test_time["minute_of_day"].values

logger.info("Columnas temporales agregadas correctamente")

2026-04-16 01:56:36,991 | INFO | Paso 5: Reconstruyendo alineación temporal desde df_test_z...
2026-04-16 01:56:37,054 | INFO | Filas df_test_z original: 99645 | filas alineadas para ventanas: 93990 | filas df_pred: 93990
2026-04-16 01:56:37,057 | INFO | Columnas temporales agregadas correctamente


In [33]:
df_pred

,y_true,y_pred,proba_-1,proba_0,proba_1,confidence,date,minute_of_day
0,1,0,0.111425,0.784246,0.104330,0.784246,2024-08-22,359
1,1,0,0.114770,0.781883,0.103347,0.781883,2024-08-22,360
2,1,0,0.112376,0.789081,0.098543,0.789081,2024-08-22,361
3,1,0,0.108022,0.788635,0.103343,0.788635,2024-08-22,362
4,0,0,0.114453,0.784984,0.100563,0.784984,2024-08-22,363
...,...,...,...,...,...,...,...,...
93985,-1,0,0.299770,0.411144,0.289085,0.411144,2025-06-13,836
93986,-1,0,0.286434,0.438782,0.274783,0.438782,2025-06-13,837
93987,-1,0,0.282306,0.444097,0.273596,0.444097,2025-06-13,838
93988,-1,0,0.274989,0.453368,0.271643,0.453368,2025-06-13,839


In [34]:
logger.info("Paso 6: Ordenando por fecha y minuto...")

df_pred = df_pred.sort_values(["date", "minute_of_day"]).reset_index(drop=True)

# Validación fuerte
is_sorted = (
    df_pred[["date", "minute_of_day"]]
    .equals(
        df_pred[["date", "minute_of_day"]]
        .sort_values(["date", "minute_of_day"])
        .reset_index(drop=True)
    )
)

if not is_sorted:
    raise ValueError("El DataFrame no quedó correctamente ordenado")

logger.info("Ordenamiento completado y validado")

# chequeo de duplicados
if df_pred.duplicated(subset=["date", "minute_of_day"]).any():
    raise ValueError("Existen timestamps duplicados")

# chequeo de saltos raros dentro del día
diff = df_pred.groupby("date")["minute_of_day"].diff().dropna()

if not (diff > 0).all():
    raise ValueError("Orden intradiario inconsistente")

2026-04-16 01:56:37,155 | INFO | Paso 6: Ordenando por fecha y minuto...
2026-04-16 01:56:37,187 | INFO | Ordenamiento completado y validado


#### Paso 7: Validaciones básicas finales



En este último paso se realiza una revisión rápida del DataFrame final de predicciones para confirmar que quedó correctamente construido antes de comenzar el análisis del comportamiento real del modelo.

Las validaciones principales son las siguientes:

- Shape final  
  Permite verificar que la tabla tiene la cantidad esperada de filas y columnas.

- Distribución de y_true  
  Muestra cómo están distribuidas las clases reales en el conjunto de test. Esto sirve como referencia para interpretar el comportamiento del modelo.

- Distribución de y_pred  
  Muestra cómo se están distribuyendo las predicciones del modelo. Este punto es especialmente importante para detectar si el modelo está:
  - concentrándose demasiado en la clase neutral
  - generando pocas señales de entrada
  - colapsando hacia una sola clase

- Preview del DataFrame  
  Permite inspeccionar visualmente las primeras filas y verificar que las columnas clave quedaron correctamente alineadas:
  - y_true
  - y_pred
  - probabilidades por clase
  - confidence
  - date
  - minute_of_day

Opcionalmente, también es recomendable revisar:

- si existen valores nulos
- el rango y promedio de confidence

En resumen, este paso actúa como un control final de calidad antes de pasar al análisis operativo de señales.

In [35]:
# =========================================
# 7. Validaciones rápidas
# =========================================

logger.info("Paso 7: Validaciones básicas...")

logger.info(f"Shape final: {df_pred.shape}")

# -----------------------------
# Nulos
# -----------------------------
null_counts = df_pred.isnull().sum()
if null_counts.any():
    logger.warning("Se detectaron valores nulos:")
    logger.warning("\n" + null_counts[null_counts > 0].to_string())
else:
    logger.info("No se detectaron valores nulos")

# -----------------------------
# Distribución real
# -----------------------------
logger.info("Distribución y_true:")
logger.info("\n" + df_pred["y_true"].value_counts(normalize=True).sort_index().to_string())

# -----------------------------
# Distribución predicha
# -----------------------------
logger.info("Distribución y_pred:")
logger.info("\n" + df_pred["y_pred"].value_counts(normalize=True).sort_index().to_string())

# -----------------------------
# Confidence
# -----------------------------
logger.info(
    "Resumen confidence | "
    f"min={df_pred['confidence'].min():.4f} | "
    f"mean={df_pred['confidence'].mean():.4f} | "
    f"median={df_pred['confidence'].median():.4f} | "
    f"max={df_pred['confidence'].max():.4f}"
)

# -----------------------------
# Preview
# -----------------------------
logger.info("Preview del DataFrame:")
logger.info("\n" + df_pred.head().to_string())

2026-04-16 01:56:37,222 | INFO | Paso 7: Validaciones básicas...
2026-04-16 01:56:37,223 | INFO | Shape final: (93990, 8)
2026-04-16 01:56:37,229 | INFO | No se detectaron valores nulos
2026-04-16 01:56:37,229 | INFO | Distribución y_true:
2026-04-16 01:56:37,232 | INFO | 
y_true
-1    0.212076
 0    0.573774
 1    0.214150
2026-04-16 01:56:37,233 | INFO | Distribución y_pred:
2026-04-16 01:56:37,239 | INFO | 
y_pred
-1    0.198266
 0    0.574242
 1    0.227492
2026-04-16 01:56:37,241 | INFO | Resumen confidence | min=0.3336 | mean=0.4914 | median=0.4384 | max=0.9039
2026-04-16 01:56:37,242 | INFO | Preview del DataFrame:
2026-04-16 01:56:37,246 | INFO | 
   y_true  y_pred  proba_-1   proba_0   proba_1  confidence        date  minute_of_day
0       1       0  0.111425  0.784246  0.104330    0.784246  2024-08-22            359
1       1       0  0.114770  0.781883  0.103347    0.781883  2024-08-22            360
2       1       0  0.112376  0.789081  0.098543    0.789081  2024-08-22    

### **3. Validaciones clave (obligatorias antes de continuar)**


Antes de avanzar al análisis del comportamiento del modelo, es fundamental validar que el DataFrame de predicciones (`df_pred`) es consistente desde el punto de vista temporal, estadístico y probabilístico.

Orden temporal correcto

Las observaciones deben estar estrictamente ordenadas por:
- date
- minute_of_day

Esto garantiza que no existe mezcla de días ni desalineación temporal entre las predicciones y el dataset original. Esta condición es crítica, ya que cualquier análisis posterior (especialmente en trading) depende de la correcta secuencia temporal.

Distribución de clases

Se debe comparar la distribución de:
- y_true (real)
- y_pred (predicho)

El objetivo es verificar que el modelo:
- no esté colapsando en una sola clase (especialmente la clase 0)
- esté generando predicciones en ambas direcciones (-1 y 1)

Un modelo que predice mayoritariamente 0 no es útil operativamente, aunque tenga métricas aceptables.

Probabilidades coherentes

La variable `confidence`, definida como el máximo de las probabilidades por clase, debe cumplir:

- valores en el rango [0.33, 1.0]
- presencia de variabilidad (no constante)

Interpretación:
- ~0.33 → el modelo está indeciso (comportamiento cercano a aleatorio)
- valores altos (>0.6–0.7) → señales con mayor confianza

Si todas las observaciones tienen confidence cercana a 0.33, el modelo no está capturando señal útil.


### **4. Resultado esperado**

Al finalizar este proceso, se debe contar con un DataFrame estructurado de la siguiente forma:

y_true | y_pred | proba_-1 | proba_0 | proba_1 | confidence | date | minute_of_day

Cada fila representa una predicción del modelo alineada con su contexto temporal y su distribución probabilística.

Ejemplo:

y_true	y_pred	proba_-1	proba_0	proba_1	confidence	date	minute_of_day
0	0	0.10	0.75	0.15	0.75	...	...
1	1	0.20	0.30	0.50	0.50	...	...

Este formato es el mínimo necesario para poder analizar el modelo desde una perspectiva operativa.


### **5. Punto importante (conceptual)**


Este paso marca la transición más importante del flujo de trabajo.

Hasta ahora, el modelo se evaluaba mediante métricas globales (balanced accuracy, F1, etc.), que resumen el comportamiento en un único valor.

A partir de este punto, se trabaja con:

- decisiones individuales (fila a fila)
- contexto temporal real
- nivel de confianza asociado a cada predicción

Esto permite pasar de un análisis abstracto a un análisis operativo.

En particular, este dataset habilita:

- detectar señales de trading
- medir la frecuencia de oportunidades
- analizar el comportamiento intradía del modelo
- evaluar cuándo el modelo es más confiable

En otras palabras, el modelo deja de ser una “caja negra evaluada con métricas” y pasa a ser un generador de decisiones en el tiempo.

---

Con esto completo, se da por finalizada la reconstrucción del set de predicciones.

El siguiente paso es:

Punto 2 → definición y análisis de la variable `confidence`, como base para filtrar señales y evaluar la calidad real del modelo.

## **11.2. Validar desempeño real a nivel de predicción**

En este punto se vuelve a evaluar el desempeño del modelo, pero ya no a partir de los outputs del pipeline de entrenamiento, sino directamente sobre la tabla final de predicciones (`df_pred`).

Esto es importante porque `df_pred` representa el comportamiento real del modelo en condiciones operativas, con:

- predicciones alineadas temporalmente
- probabilidades correctamente interpretadas
- estructura lista para análisis de trading

Las métricas a recalcular son:

- balanced accuracy  
- f1 macro  
- accuracy  

Estas métricas permiten confirmar que los resultados observados previamente se mantienen en el dataset final.

Además, se incorporan dos análisis clave:

- Matriz de confusión

  Permite visualizar en detalle cómo se distribuyen los aciertos y errores del modelo entre las clases:

  - cuántas veces acierta cada clase
  - en qué clases se equivoca más
  - si tiende a confundir direcciones (ej. predice neutral cuando hay movimiento)

  Este análisis es fundamental porque las métricas globales pueden ocultar comportamientos específicos del modelo.

- Accuracy por clase

  Se calcula la precisión del modelo por cada clase individual:

  - accuracy para -1 (bajadas)
  - accuracy para 0 (neutral)
  - accuracy para 1 (subidas)

  Esto permite identificar si el modelo funciona mejor en ciertos escenarios del mercado y peor en otros.

---

**Objetivo**

El objetivo de este paso es entender:

- dónde acierta el modelo
- dónde falla
- qué tipo de errores comete

En el contexto de trading, este diagnóstico es crítico, ya que no todos los errores tienen el mismo impacto.

Un modelo puede tener métricas globales aceptables, pero si falla sistemáticamente en las señales de entrada (por ejemplo, confundiendo subidas con neutral), su utilidad operativa se reduce significativamente.

Por lo tanto, este paso permite pasar de una evaluación cuantitativa global a una comprensión cualitativa del comportamiento del modelo.

In [36]:
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)
import pandas as pd
import numpy as np

# =========================================
# 11.2 Validar desempeño real a nivel de predicción
# =========================================

logger.info("11.2 | Validando desempeño real del modelo sobre df_pred...")

# -----------------------------
# 1. Extraer valores reales y predichos
# -----------------------------
y_true = df_pred["y_true"].to_numpy()
y_pred = df_pred["y_pred"].to_numpy()

# Orden fijo de clases
labels = [-1, 0, 1]

# -----------------------------
# 2. Métricas globales
# -----------------------------
bal_acc = balanced_accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average="macro")
acc = accuracy_score(y_true, y_pred)

logger.info(f"Balanced Accuracy : {bal_acc:.6f}")
logger.info(f"F1 Macro          : {f1_macro:.6f}")
logger.info(f"Accuracy          : {acc:.6f}")

# -----------------------------
# 3. Matriz de confusión
# -----------------------------
cm = confusion_matrix(y_true, y_pred, labels=labels)

df_cm = pd.DataFrame(
    cm,
    index=[f"real_{c}" for c in labels],
    columns=[f"pred_{c}" for c in labels],
)

logger.info("Matriz de confusión:")
logger.info("\n" + df_cm.to_string())

# -----------------------------
# 4. Accuracy por clase
#    (recall por clase = aciertos / total real de esa clase)
# -----------------------------
class_accuracy = {}

for i, cls in enumerate(labels):
    total_real_cls = cm[i, :].sum()
    correct_cls = cm[i, i]
    acc_cls = correct_cls / total_real_cls if total_real_cls > 0 else np.nan
    class_accuracy[cls] = acc_cls

df_class_acc = pd.DataFrame({
    "class": list(class_accuracy.keys()),
    "accuracy_by_class": list(class_accuracy.values())
})

logger.info("Accuracy por clase:")
logger.info("\n" + df_class_acc.to_string(index=False))

# -----------------------------
# 5. Reporte detallado opcional
# -----------------------------
report_dict = classification_report(
    y_true,
    y_pred,
    labels=labels,
    output_dict=True,
    zero_division=0,
)

df_report = pd.DataFrame(report_dict).T

logger.info("Classification report:")
logger.info("\n" + df_report.to_string())

# -----------------------------
# 6. Resumen final en un DataFrame
# -----------------------------
df_metrics_pred = pd.DataFrame([{
    "balanced_accuracy": bal_acc,
    "f1_macro": f1_macro,
    "accuracy": acc,
    "acc_class_-1": class_accuracy[-1],
    "acc_class_0": class_accuracy[0],
    "acc_class_1": class_accuracy[1],
    "n_samples": len(df_pred),
}])

logger.info("Resumen final de métricas sobre df_pred:")
logger.info("\n" + df_metrics_pred.to_string(index=False))

2026-04-16 01:56:37,258 | INFO | 11.2 | Validando desempeño real del modelo sobre df_pred...
2026-04-16 01:56:37,317 | INFO | Balanced Accuracy : 0.440185
2026-04-16 01:56:37,317 | INFO | F1 Macro          : 0.439733
2026-04-16 01:56:37,318 | INFO | Accuracy          : 0.536525
2026-04-16 01:56:37,354 | INFO | Matriz de confusión:
2026-04-16 01:56:37,357 | INFO | 
         pred_-1  pred_0  pred_1
real_-1     5451    8419    6063
real_0      7319   38134    8476
real_1      5865    7420    6843
2026-04-16 01:56:37,359 | INFO | Accuracy por clase:
2026-04-16 01:56:37,361 | INFO | 
 class  accuracy_by_class
    -1           0.273466
     0           0.707115
     1           0.339974
2026-04-16 01:56:37,378 | INFO | Classification report:
2026-04-16 01:56:37,381 | INFO | 
              precision    recall  f1-score       support
-1             0.292514  0.273466  0.282670  19933.000000
0              0.706538  0.707115  0.706827  53929.000000
1              0.320036  0.339974  0.329704  2

Observaciones sobre el desempeño real del modelo (11.2)

Los resultados obtenidos confirman que el pipeline es consistente y que las métricas calculadas sobre `df_pred` coinciden con las obtenidas previamente durante la fase de evaluación. Esto valida que la reconstrucción del set de predicciones fue correcta y que el análisis posterior se basa en datos confiables.

A nivel global, el modelo presenta:

- balanced_accuracy ≈ 0.44  
- f1_macro ≈ 0.44  
- accuracy ≈ 0.54  

Estas métricas indican que el modelo tiene capacidad predictiva por encima del baseline, pero no de forma uniforme entre clases.

Al analizar el desempeño por clase, se observa un comportamiento claramente asimétrico:

- Clase 0 (neutral): accuracy ≈ 0.71  
- Clase -1 (bajada): accuracy ≈ 0.27  
- Clase 1 (subida): accuracy ≈ 0.34  

Esto indica que el modelo es significativamente mejor detectando situaciones sin movimiento relevante (neutral), mientras que tiene dificultades para identificar correctamente los movimientos direccionales.

La matriz de confusión refuerza este diagnóstico. En particular:

- Cuando el mercado cae (real = -1), el modelo frecuentemente predice neutral o incluso subida.
- Cuando el mercado sube (real = 1), nuevamente el modelo tiende a predecir neutral o confundirse con la dirección opuesta.

El patrón dominante es que, ante incertidumbre, el modelo tiende a predecir la clase neutral. Esto sugiere un comportamiento conservador, donde el modelo prefiere evitar tomar una decisión direccional antes que arriesgarse a equivocarse.

Desde el punto de vista operativo, esto tiene una implicancia clara:

- El modelo es eficaz como filtro de mercado, es decir, para identificar zonas donde no conviene operar.
- Sin embargo, es menos eficaz como generador directo de señales de entrada, ya que no captura con suficiente precisión los movimientos direccionales.

La métrica de balanced accuracy refleja este comportamiento. Su valor (~0.44) surge de un promedio entre:

- una clase con muy buen desempeño (neutral)
- dos clases con desempeño bajo (direccionales)

Esto confirma que el modelo no tiene un comportamiento homogéneo, sino que está especializado en detectar ausencia de señal más que presencia de señal.

En consecuencia, el modelo no debe interpretarse como un sistema de trading directo, sino como un componente dentro de un sistema más amplio, cuya función principal es filtrar el mercado y reducir el ruido.

Finalmente, estos resultados sugieren que la señal útil del modelo no está distribuida uniformemente en todas las predicciones, sino concentrada en un subconjunto de ellas. Esto motiva el siguiente paso del análisis:

analizar la variable `confidence` para identificar en qué condiciones el modelo es realmente confiable y cómo filtrar las predicciones para obtener señales de mayor calidad.